# Test run (frozen settings) -- one model per machine
- **Calista:** `MODEL = "qwen2.5:1.5b"` | **teammate:** `MODEL = "llama3.2:3b"`
- Edit **only** the S0 cell. Everything else is the frozen pilot code (KB v2, original prompt, budget 120 words, temperature 0, seed 42).
- Run order: Restart kernel -> run all cells top to bottom. First with `TEST_SMOKE = True` (10 captions, ~5 min), then set `TEST_SMOKE = False` and run all again.
- Generation (G1) and SummaC (M1) are **resumable**: after a crash, restart and run all again; nothing saved is lost or redone.
- Send back the files that the last cell lists (results/test/...).
- **Crash-safe:** every answer and score is on disk the moment it is made. After any stop, just Restart & Run All. G1 / M1 print whether the last run stopped unexpectedly and continue from there. Status: `results/test/*_status.json`.

## S0 Settings (edit here only)

In [1]:
# S0 -- settings for this machine. The ONLY cell you edit. Peak RAM: ~0 MB
from pathlib import Path
import os, sys, json, csv, time, platform
import urllib.request

MODEL      = "qwen2.5:1.5b"   # Calista: "qwen2.5:1.5b" | teammate: "llama3.2:3b"
TEST_SMOKE = True             # True = first 10 test captions as a check; then False for the full run

# Repo root = the folder that holds data/ and notebooks/ (this notebook lives in notebooks/)
project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert MODEL in {"qwen2.5:1.5b", "llama3.2:3b"}, MODEL

KB_OUT       = project_dir/"data"/"subtask_kb_build_v2"
TEST_PARQUET = project_dir/"data"/"MedLayBench-V"/"chest"/"test_chest_text_final.parquet"
TEST_GOLD    = project_dir/"data"/"subtask_gold_concepts_det"/"mlbv_chest_test_gold.tsv"
TEST_MENT    = project_dir/"data"/"subtask_gold_concepts_det"/"mlbv_chest_test_mentions.tsv"
need = [KB_OUT/"kb_manifest.json", KB_OUT/"kb_docs.jsonl", KB_OUT/"kg_edges.tsv",
        KB_OUT/"faiss_minilm"/"index.faiss", KB_OUT/"faiss_minilm"/"stamp.txt",
        TEST_PARQUET, TEST_GOLD, TEST_MENT]
missing = [str(p) for p in need if not p.exists()]
assert not missing, "unzip the data package into the repo root first; missing:\n" + "\n".join(missing)

# Ask the running Ollama server (local only, 10 s limit) which models it has. Start the Ollama app first.
try:
    with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=10) as r:
        tags = json.load(r)["models"]
except OSError as e:
    raise RuntimeError("Ollama is not running: open the Ollama app, wait ~10 s, then run S0 again") from e

# Expected Ollama model IDs (from Calista's machine, 2026-11-21); both machines must use the same weights
EXPECTED_ID = {"qwen2.5:1.5b": "65ec06548149", "llama3.2:3b": "a80c4f17acd5"}
found = {m["name"]: m["digest"][:12] for m in tags}
assert MODEL in found, f"model not pulled: run  ollama pull {MODEL}  (found: {sorted(found)})"
assert found[MODEL] == EXPECTED_ID[MODEL], \
    f"{MODEL} has ID {found[MODEL]}, expected {EXPECTED_ID[MODEL]}: different weights, tell Calista"

TAG = {"qwen2.5:1.5b": "qwen25_1.5b", "llama3.2:3b": "llama32_3b"}[MODEL]
OUT_DIR = project_dir/"results"/"test"
OUT_DIR.mkdir(parents=True, exist_ok=True)
gen_file    = OUT_DIR/f"test_kb_v2_{TAG}_generations.jsonl"
summac_file = OUT_DIR/f"test_kb_v2_{TAG}_summac.jsonl"

machine = {"model": MODEL, "ollama_id": found[MODEL], "os": platform.platform(),
           "cpu": platform.processor(), "cpu_count": os.cpu_count(), "python": sys.version.split()[0],
           "recorded": time.strftime("%Y-%m-%d %H:%M")}
(OUT_DIR/f"machine_{TAG}.json").write_text(json.dumps(machine, indent=2), encoding="utf-8")
print(json.dumps(machine, indent=2))
print("smoke run (10 captions)" if TEST_SMOKE else "FULL test run")

{
  "model": "qwen2.5:1.5b",
  "ollama_id": "65ec06548149",
  "os": "Windows-10-10.0.26300-SP0",
  "cpu": "Intel64 Family 6 Model 141 Stepping 1, GenuineIntel",
  "cpu_count": 12,
  "python": "3.10.21",
  "recorded": "2026-11-21 17:24"
}
smoke run (10 captions)


In [2]:
# R0 -- crash-safe saving, used by G1 (generation) and M1 (SummaC). Peak RAM: ~0 MB
# 1. Every answer / score is one line, flushed AND fsynced to disk at once: a crash loses at most the line in progress.
# 2. On start, a half-written last line (power cut mid-write) is found and removed; it is simply redone.
# 3. A status file records progress every 10 lines. If it still says "running" when you start again,
#    the previous run was stopped unexpectedly; the cell says so and resumes from the saved file.
# 4. A backup copy (.bak) of the output file is made every BACKUP_EVERY lines and at the end.
# 5. Windows is kept awake while G1 / M1 run (closing the laptop lid can still sleep it: don't).
import shutil, ctypes

BACKUP_EVERY = 500

def read_jsonl_safe(path):
    """All complete lines of a JSONL file. A broken LAST line is cut off; a broken line elsewhere stops the run."""
    if not path.exists():
        return []
    raw = path.read_bytes()
    *complete, tail = raw.split(b"\n")             # tail = text after the last newline (normally empty)
    rows = []
    for n, line in enumerate(complete, start=1):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            raise RuntimeError(f"{path.name} line {n} is damaged. Copy {path.name}.bak over it, "
                               f"then run again (only lines after the backup are redone).")
    if tail.strip():                               # last line was only partly written when the run stopped
        with open(path, "r+b") as f:
            f.truncate(len(raw) - len(tail))
        print(f"repaired {path.name}: removed 1 half-written last line; it will be redone")
    return rows

class SafeAppender:
    """Append JSON lines; each one is on disk before the next answer starts."""
    def __init__(self, path):
        self.path, self.n = path, 0
        self.bak = path.with_name(path.name + ".bak")
        self.f = open(path, "a", encoding="utf-8", newline="\n")
    def write(self, rec):
        self.f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        self.f.flush()
        os.fsync(self.f.fileno())
        self.n += 1
        if self.n % BACKUP_EVERY == 0:
            shutil.copy2(self.path, self.bak)
    def close(self):
        self.f.close()
        if self.n:
            shutil.copy2(self.path, self.bak)

def status_file(stage):
    return OUT_DIR/f"test_kb_v2_{TAG}_{stage}_status.json"

def write_status(stage, **info):
    """Atomic write (temp file + rename), so the status file itself is never half-written."""
    p = status_file(stage)
    tmp = p.with_name(p.name + ".tmp")
    tmp.write_text(json.dumps(info | {"model": MODEL, "smoke": TEST_SMOKE,
                                      "updated": time.strftime("%Y-%m-%d %H:%M:%S")}, indent=2), encoding="utf-8")
    os.replace(tmp, p)

def report_previous(stage):
    p = status_file(stage)
    if not p.exists():
        print(f"[{stage}] first start")
        return
    s = json.loads(p.read_text(encoding="utf-8"))
    if s["state"] == "running":
        print(f"[{stage}] PREVIOUS RUN STOPPED UNEXPECTEDLY (crash, power cut, PC restart or closed window) "
              f"at {s['updated']} after {s['done']:,}/{s['total']:,}. Resuming from the saved file.")
    else:
        print(f"[{stage}] previous run: {s['state']} at {s['updated']} ({s['done']:,}/{s['total']:,})")

def keep_awake(on):
    """Ask Windows not to sleep while a long cell runs."""
    if os.name == "nt":
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000001 if on else 0x80000000)

# self-check: a file cut off mid-line is repaired, complete lines are kept
_t = OUT_DIR/"_selfcheck.jsonl"
_t.write_bytes(b'{"a": 1}\n{"a": 2}\n{"a": ')
assert read_jsonl_safe(_t) == [{"a": 1}, {"a": 2}] and _t.read_bytes() == b'{"a": 1}\n{"a": 2}\n'
_t.unlink()
print("crash-safe saving ready | backup every", BACKUP_EVERY, "lines | status files in", OUT_DIR)

repaired _selfcheck.jsonl: removed 1 half-written last line; it will be redone
crash-safe saving ready | backup every 500 lines | status files in d:\Project\Radiology_RAG\results\test


## KB, graph and retrievers (frozen)

In [3]:
# Cell 1 -- load the real KB (A1 variant) built in build_knowledge.ipynb. Peak RAM: ~100 MB
from pathlib import Path
import json, csv

KB_OUT = project_dir/"data"/"subtask_kb_build_v2"

manifest = json.loads((KB_OUT/"kb_manifest.json").read_text(encoding="utf-8"))
with open(KB_OUT/"kb_docs.jsonl", encoding="utf-8") as f:
    kb_concepts = [json.loads(line) for line in f]          # one dict per CUI: cui, name, synonyms, definition, ...
with open(KB_OUT/"kg_edges.tsv", encoding="utf-8", newline="") as f:
    kb_edges = [(r["child"], r["parent"], r["rel"]) for r in csv.DictReader(f, delimiter="\t")]

assert len(kb_concepts) == manifest["kb_cuis"], "kb_docs.jsonl does not match the manifest"
assert len(kb_edges) == manifest["kg_edges"], "kg_edges.tsv does not match the manifest"
print(f"KB variant: {manifest['kb_variant']} | built {manifest['built']}")
print(f"concepts: {len(kb_concepts):,} | edges: {len(kb_edges):,} | "
      f"gold_hard ceiling (micro): {manifest['ceiling']['hard_micro']:.3f}")

KB variant: A1 seeds + hierarchy neighbours | built 2026-11-16 22:29
concepts: 33,863 | edges: 162,221 | gold_hard ceiling (micro): 0.537


In [4]:
# Cell 3 -- KB documents (capped length) + knowledge graph. Peak RAM: ~300 MB
import re
import networkx as nx
from langchain_core.documents import Document

MAX_DOC_WORDS  = 60   # full docs: median 18, p90 54, 311 docs > 120 words; 60 keeps >= 2 docs inside the 120-word budget
MAX_HEAD_WORDS = 35   # name + synonyms may use at most this much; the rest goes to the definition
MAX_SYN_IN_DOC = 5    # synonyms are stored shortest-first, so the 5 shortest are used

def fit_definition(definition, room):
    """Whole sentences until `room` words are used; if the first sentence alone is too long, cut it with '...'."""
    if room <= 0 or not definition:
        return ""
    if len(definition.split()) <= room:
        return definition
    kept = []
    for sent in re.split(r"(?<=[.!?])\s+", definition):
        w = sent.split()
        if len(kept) + len(w) > room:
            break
        kept += w
    return " ".join(kept) if kept else " ".join(definition.split()[:room]) + "..."

def concept_to_doc(c):
    syns = []
    for s in c["synonyms"][:MAX_SYN_IN_DOC]:
        if len(c["name"].split()) + 2 + len(" ".join(syns + [s]).split()) <= MAX_HEAD_WORDS:
            syns.append(s)
    head = c["name"] + "." + (" Also called: " + ", ".join(syns) + "." if syns else "")
    definition = fit_definition(c["definition"], MAX_DOC_WORDS - len(head.split()) - 1)  # 1 word for "Definition:"
    text = head + (" Definition: " + definition if definition else "")
    return Document(page_content=text, metadata={"cui": c["cui"], "name": c["name"],
                                                 "def_source": c["def_source"], "kb_role": c["kb_role"]})

kb_docs = [concept_to_doc(c) for c in kb_concepts]
doc_by_cui = {d.metadata["cui"]: d for d in kb_docs}
doc_words = sorted(len(d.page_content.split()) for d in kb_docs)
print(f"docs: {len(kb_docs):,} | words median {doc_words[len(doc_words)//2]}, max {doc_words[-1]} | "
      f"over {MAX_DOC_WORDS}: {sum(w > MAX_DOC_WORDS for w in doc_words)} (only if the name alone is very long)")

kg = nx.DiGraph()
for c in kb_concepts:
    kg.add_node(c["cui"], name=c["name"])
for child, parent, rel in kb_edges:
    kg.add_edge(child, parent, rel=rel)
del kb_edges                                   # graph holds the edges now

kg_undirected = kg.to_undirected(as_view=True) #hops go up AND down
kg_degree = dict(kg_undirected.degree())       # used to rank specific concepts above hubs in k-hop
assert set(kg.nodes) == set(doc_by_cui), "graph nodes and KB documents differ"
print(kg)

docs: 33,863 | words median 16, max 60 | over 60: 0 (only if the name alone is very long)
DiGraph with 33863 nodes and 162221 edges


In [5]:
# Cell 5 -- BM25 + dense (FAISS) search. First run embeds 43k docs (~5-15 min CPU, estimate) and saves the index.
# Peak RAM: ~400 MB (embedding model + index + BM25)
import re, numpy as np
from rank_bm25 import BM25Okapi
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    encode_kwargs={"normalize_embeddings": True} # L2 ranking == cosine ranking
    )

# Cache: rebuild only if the KB or the document format changed
FAISS_DIR = KB_OUT/"faiss_minilm"
stamp = f"{manifest['built']}|{len(kb_docs)}|{MAX_DOC_WORDS}|{MAX_HEAD_WORDS}|{MAX_SYN_IN_DOC}"
stamp_file = FAISS_DIR/"stamp.txt"
if stamp_file.exists() and stamp_file.read_text(encoding="utf-8") == stamp:
    # our own file, so loading its pickled docstore is safe
    faiss_store = FAISS.load_local(str(FAISS_DIR), embeddings, allow_dangerous_deserialization=True)
    print("FAISS index loaded from", FAISS_DIR)
else:
    faiss_store = FAISS.from_documents(kb_docs, embeddings)
    faiss_store.save_local(str(FAISS_DIR))
    stamp_file.write_text(stamp, encoding="utf-8")
    print("FAISS index built and saved to", FAISS_DIR)
assert faiss_store.index.ntotal == len(kb_docs)

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower()) #lowercase, drop punctuation
bm25_index = BM25Okapi([tokenize(d.page_content) for d in kb_docs])

def bm25_search(query, k):
    scores = bm25_index.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [kb_docs[i] for i in top if scores[i] > 0] #no zero-overlap docs

def dense_search(query, k):
    return faiss_store.similarity_search(query, k=k)

def rrf_fuse(ranked_lists, k=60, top_n = 5):
    """Reciprocal Rank Fussion: score(d) = sum (1/(k + rank_i(d))"""

    scores, docs = {}, {}
    for ranked in ranked_lists:
        for rank, d in enumerate(ranked, start = 1):
            cui = d.metadata["cui"]
            scores[cui] = scores.get(cui, 0.0) + 1.0/(k + rank)
            docs[cui] = d
    best = sorted(scores, key=scores.get, reverse=True)[:top_n]

    return [docs[c] for c in best]

def retrieve_hybrid(inputs, top_n=5, pool=10):
    query = inputs["jargon"]

    return rrf_fuse([bm25_search(query, pool), dense_search(query, pool)], top_n=top_n)

C:\Users\calis\AppData\Local\Temp\ipykernel_31176\3476422662.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
c:\anaconda3\envs\kg-rag\lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


FAISS index loaded from d:\Project\Radiology_RAG\data\subtask_kb_build_v2\faiss_minilm


In [6]:
# Cell 7 -- k-hop retrieval from the anchor CUIs. Peak RAM: < 20 MB per call
from collections import Counter

def retrieve_khop(inputs, hops=1, max_nodes=50):
    """Rank: hop distance, then shared by more anchors, then fewer links (specific before hub), then CUI."""
    dist, n_anchor = {}, Counter()
    for anchor in dict.fromkeys(inputs["cuis"]):     # drop duplicate anchors, keep order
        if anchor not in kg_undirected:              # anchor outside the KB: nothing to expand
            continue
        reach = nx.single_source_shortest_path_length(kg_undirected, anchor, cutoff=hops)
        for cui, d in reach.items():
            dist[cui] = min(d, dist.get(cui, d))
            n_anchor[cui] += 1
    ranked = sorted(dist, key=lambda c: (dist[c], -n_anchor[c], kg_degree[c], c))[:max_nodes]
    return [doc_by_cui[c] for c in ranked]

def retrieve_1hop(inputs):
    return retrieve_khop(inputs, hops=1)

def retrieve_2hop(inputs):
    return retrieve_khop(inputs, hops=2)

In [7]:
CONTEXT_WORD_BUDGET = 120

def n_words(text):
    return len(text.split())

def apply_budget(docs, budget=CONTEXT_WORD_BUDGET):
    """Keep whole docs in rank order until the budget is full; stop at the first doc that doesn't fit."""
    kept, used, seen = [], 0, set()
    for d in docs:
        cui = d.metadata["cui"]
        if cui in seen:
            continue
        w = n_words(d.page_content)
        if used + w > budget:
            break
        kept.append(d)
        used += w
        seen.add(cui)
    return kept

def retrieve_oracle(inputs):
    """Upper bound: only the definitions of the gold CUIs."""
    return [doc_by_cui[c] for c in inputs["cuis"] if c in doc_by_cui]

def retrieve_hybrid_anchored(inputs, top_n=50, pool=50):
    """Hybrid search that also sees the anchor concept names, same entry info as the graph methods."""
    names = [doc_by_cui[c].metadata["name"] for c in inputs["cuis"] if c in doc_by_cui]
    query = inputs["jargon"] + " " + " ".join(names)
    return rrf_fuse([bm25_search(query, pool), dense_search(query, pool)], top_n=top_n)

## Prompt and chains (frozen)

In [8]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

slm_names = [MODEL]   # one model per machine (set in S0)

# Answer-length cap. Smoke + pilot answers: median 32 words, longest normal one 111 words.
# One answer looped to 30,179 words (28 min). 256 tokens (~190 words) ends such loops but leaves normal answers untouched.
MAX_ANSWER_TOKENS = 256
llms = {
    m: init_chat_model(m, model_provider="ollama", temperature=0, seed=42, num_ctx=4096,
                       num_predict=MAX_ANSWER_TOKENS)
    for m in slm_names
}

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You rewrite radiology report text for patients with no medical training. "
     "Use the reference concepts only to understand the terms. Keep every finding. "
     "Do not add diagnoses, advice, or facts that are not in the original. Use short, plain sentences."),
    ("human", "Reference concepts:\n{context}\n\nOriginal text:\n{jargon}\n\nPlain-language version:"),
])

def format_docs(docs):
    return "\n".join(f"- [{d.metadata['cui']}] {d.page_content}" for d in docs) or "(none)"

def build_chain(retrieve_fn, llm):
    #Output keeps the retrieved docs too: for evaluation purposes.

    return (
        RunnablePassthrough.assign(docs=RunnableLambda(lambda x: apply_budget(retrieve_fn(x))))
        .assign(context=lambda x: format_docs(x["docs"]))
        .assign(answer=prompt | llm | StrOutputParser())
    )

retrievers = {
    "no_rag":          lambda x: [],
    "oracle":          retrieve_oracle,
    "hybrid":          lambda x: retrieve_hybrid(x, top_n=50, pool=50),
    "hybrid_anchored": retrieve_hybrid_anchored,
    "1hop":            retrieve_1hop,
    "2hop":            retrieve_2hop,
}

chains = {(m, r): build_chain(fn, llms[m]) for m in slm_names for r, fn in retrievers.items()}

In [9]:
for key, chain in chains.items():
    llm_used = chain.steps[-1].mapper.steps__["answer"].steps[1]
    print(key, type(llm_used).__name__, getattr(llm_used, "model", None))

('qwen2.5:1.5b', 'no_rag') ChatOllama qwen2.5:1.5b
('qwen2.5:1.5b', 'oracle') ChatOllama qwen2.5:1.5b
('qwen2.5:1.5b', 'hybrid') ChatOllama qwen2.5:1.5b
('qwen2.5:1.5b', 'hybrid_anchored') ChatOllama qwen2.5:1.5b
('qwen2.5:1.5b', '1hop') ChatOllama qwen2.5:1.5b
('qwen2.5:1.5b', '2hop') ChatOllama qwen2.5:1.5b


## Test captions

In [10]:
# T1 -- test captions + gold concepts. Same exclusion rule as the pilot. Peak RAM: ~20 MB
import pandas as pd

tst = pd.read_parquet(TEST_PARQUET, columns=["expert_caption", "lay_caption"])
with open(TEST_GOLD, encoding="utf-8", newline="") as f:
    gold_rows = list(csv.DictReader(f, delimiter="\t"))
assert len(gold_rows) == len(tst), f"gold rows {len(gold_rows)} != captions {len(tst)}"

samples_by_id, n_empty = {}, 0
for r in gold_rows:
    i = int(r["row"])
    gold_all = [c for c in r["gold_all"].split(";") if c]
    if not gold_all:                      # no gold concept: nothing to anchor or score, excluded
        n_empty += 1
        continue
    samples_by_id[i] = {"id": i,
                        "jargon": tst.iloc[i]["expert_caption"],     # source caption
                        "lay": tst.iloc[i]["lay_caption"],           # MedLayBench-V lay reference
                        "cuis": gold_all,                            # anchors for oracle / anchored / k-hop
                        "gold_hard": [c for c in r["gold_hard"].split(";") if c]}
del tst, gold_rows
assert len(samples_by_id) == 2790, f"expected 2,790 usable test captions, got {len(samples_by_id)}"

RUN_IDS = sorted(samples_by_id)[:10] if TEST_SMOKE else sorted(samples_by_id)
n_hard = sum(len(samples_by_id[i]["gold_hard"]) for i in RUN_IDS)
print(f"test captions: {len(samples_by_id) + n_empty:,} | empty gold (excluded): {n_empty} | "
      f"usable: {len(samples_by_id):,} | this run: {len(RUN_IDS):,}{' (SMOKE)' if TEST_SMOKE else ''}")
print(f"gold_hard in KB: {sum(c in doc_by_cui for i in RUN_IDS for c in samples_by_id[i]['gold_hard'])}/{n_hard}")

test captions: 2,796 | empty gold (excluded): 6 | usable: 2,790 | this run: 10 (SMOKE)
gold_hard in KB: 13/21


## Generation

In [11]:
# G1 -- generate every method's answer for every caption. Crash-safe and resumable (see R0).
# Full run: 2,790 captions x 6 methods = 16,740 answers (qwen ~2.5 h, llama ~7 h at pilot speed).
# Peak RAM: the loaded Ollama model (~1-3 GB) + < 50 MB here
CAP_FLAG_WORDS = 170   # an answer this long probably hit MAX_ANSWER_TOKENS; counted, not removed
RETRIES, RETRY_WAIT = 4, 60   # if Ollama hiccups: wait 60 s and retry, up to 3 times, before stopping

report_previous("generation")
done = {(r["model"], r["method"], r["sample_id"]) for r in read_jsonl_safe(gen_file)}
total = len(RUN_IDS) * len(retrievers)
n_have = sum((MODEL, m, sid) in done for sid in RUN_IDS for m in retrievers)
print(f"already saved: {n_have:,}/{total:,} answers in {gen_file.name}")

def hard_recall(method, docs, s):
    """Share of the caption's gold_hard CUIs in the context the model saw. None for no_rag / no gold_hard."""
    if method == "no_rag" or not s["gold_hard"]:
        return None
    got = {d.metadata["cui"] for d in docs}
    return sum(c in got for c in s["gold_hard"]) / len(s["gold_hard"])

def invoke_with_retry(chain, x):
    for t in range(1, RETRIES + 1):
        try:
            return chain.invoke(x)
        except Exception as e:
            if t == RETRIES:
                raise
            print(f"  {time.strftime('%H:%M')} Ollama call failed ({type(e).__name__}: {str(e)[:80]}); "
                  f"retry {t}/{RETRIES - 1} in {RETRY_WAIT} s")
            time.sleep(RETRY_WAIT)

PRINT_EVERY = 1 if TEST_SMOKE else 100
t_start, n_new, sid = time.time(), 0, None
out = SafeAppender(gen_file)
keep_awake(True)
write_status("generation", state="running", done=n_have, total=total, last_sample_id=None)
try:
    for k, sid in enumerate(RUN_IDS, start=1):
        s = samples_by_id[sid]
        for method in retrievers:
            if (MODEL, method, sid) in done:
                continue
            t0 = time.time()
            res = invoke_with_retry(chains[(MODEL, method)], {"jargon": s["jargon"], "cuis": s["cuis"]})
            n_ans = len(res["answer"].split())
            out.write({"model": MODEL, "method": method, "sample_id": sid,
                       "answer": res["answer"], "answer_words": n_ans,
                       "maybe_capped": n_ans >= CAP_FLAG_WORDS,
                       "retrieved": [d.metadata["cui"] for d in res["docs"]],
                       "n_docs": len(res["docs"]),
                       "context_words": sum(n_words(d.page_content) for d in res["docs"]),
                       "gold_cui_recall": hard_recall(method, res["docs"], s),
                       "seconds": round(time.time() - t0, 2)})
            n_new += 1
            if n_new % 10 == 0:
                write_status("generation", state="running", done=n_have + n_new, total=total, last_sample_id=sid)
            if time.time() - t0 > 30:
                print(f"  slow: {method} id={sid} {time.time() - t0:.0f}s, {n_ans} words")
        if k == 1 or k % PRINT_EVERY == 0 or k == len(RUN_IDS):
            el = time.time() - t_start
            left = (total - n_have - n_new) * el / max(n_new, 1)
            print(f"{time.strftime('%H:%M')} | caption {k:,}/{len(RUN_IDS):,} | saved {n_have + n_new:,}/{total:,} | "
                  f"{el/max(n_new,1):.1f} s/answer | ~{left/3600:.1f} h left")
    write_status("generation", state="finished", done=n_have + n_new, total=total, last_sample_id=sid)
except BaseException as e:   # Interrupt button, Ollama down after retries, any error: record it, then stop
    why = "interrupted (Interrupt button)" if isinstance(e, KeyboardInterrupt) else f"error: {type(e).__name__}: {str(e)[:150]}"
    write_status("generation", state="stopped, " + why, done=n_have + n_new, total=total, last_sample_id=sid)
    print(f"\nSTOPPED at caption id {sid}: {why}\n{n_have + n_new:,}/{total:,} answers are safely saved. "
          f"Fix the cause (e.g. open Ollama), then Restart & Run All: it continues from here.")
    raise
finally:
    out.close()
    keep_awake(False)

scored = pd.DataFrame(read_jsonl_safe(gen_file))
scored = scored[(scored["model"] == MODEL) & scored["sample_id"].isin(RUN_IDS)]
scored = scored.drop_duplicates(["method", "sample_id"], keep="last").reset_index(drop=True)
assert len(scored) == total, f"{len(scored)} answers, expected {total}: run G1 again"
print(f"\n{len(scored):,} answers | probably hit the cap: {int(scored['maybe_capped'].sum())}")
display(scored.groupby("method")[["seconds", "n_docs", "context_words", "answer_words", "gold_cui_recall"]].mean().round(2))

[generation] first start
already saved: 60/60 answers in test_kb_v2_qwen25_1.5b_generations.jsonl
17:24 | caption 1/10 | saved 60/60 | 0.0 s/answer | ~0.0 h left
17:24 | caption 2/10 | saved 60/60 | 0.0 s/answer | ~0.0 h left
17:24 | caption 3/10 | saved 60/60 | 0.0 s/answer | ~0.0 h left
17:24 | caption 4/10 | saved 60/60 | 0.0 s/answer | ~0.0 h left
17:24 | caption 5/10 | saved 60/60 | 0.0 s/answer | ~0.0 h left
17:24 | caption 6/10 | saved 60/60 | 0.0 s/answer | ~0.0 h left
17:24 | caption 7/10 | saved 60/60 | 0.0 s/answer | ~0.0 h left
17:24 | caption 8/10 | saved 60/60 | 0.0 s/answer | ~0.0 h left
17:24 | caption 9/10 | saved 60/60 | 0.0 s/answer | ~0.0 h left
17:24 | caption 10/10 | saved 60/60 | 0.0 s/answer | ~0.0 h left

60 answers | probably hit the cap: 0


,seconds,n_docs,context_words,answer_words,gold_cui_recall
method,,,,,
1hop,0.30,4.1,97.9,14.7,0.63
2hop,0.28,3.4,98.3,14.7,0.69
hybrid,0.56,6.1,103.5,14.3,0.19
hybrid_anchored,0.52,5.8,110.7,13.4,0.37
no_rag,13.23,0.0,0.0,18.5,NaN
oracle,0.27,1.8,56.0,14.3,0.69


In [12]:
# P3 -- remove the model's boilerplate opener ("Here is the rewritten text:") before scoring.
# Same rule for every method; only a whole first line ending in ':' is removed. Raw text kept in answer_raw.
# Peak RAM: ~0 MB Llama
import re
OPENER = re.compile(r"^\s*(?:here is|here's|below is|sure)[^\n]*:[ \t]*\n+", re.IGNORECASE)

def clean_answer(text):
    return OPENER.sub("", text, count=1).strip()

if "answer_raw" not in scored:
    scored["answer_raw"] = scored["answer"]
scored["answer"] = scored["answer_raw"].map(clean_answer)
changed = scored["answer"] != scored["answer_raw"].str.strip()
print(f"opener removed from {changed.sum()}/{len(scored)} answers")
print(scored.loc[changed, "answer_raw"].str.split("\n").str[0].value_counts().head(5))

opener removed from 0/60 answers
Series([], Name: count, dtype: int64)


## Metrics

In [13]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

NLI_MODEL = "tals/albert-base-vitaminc-mnli"   # SummaC's "vitc-base"
nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).eval()

def find_label(names, default):
    for i, lab in nli_model.config.id2label.items():
        if lab.lower() in names:
            return int(i)
    return default

ENT_IDX = find_label({"entailment", "supports"}, 0)       # SummaC's map: vitc entail=0
CON_IDX = find_label({"contradiction", "refutes"}, 1)     # SummaC's map: vitc contradict=1
print(nli_model.config.id2label, "-> entail", ENT_IDX, "| contradict", CON_IDX)

def split_sentences(text):
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
    return sents or [text.strip()]

@torch.no_grad()
def summac_zs(premise_text, output_text, batch_size=32):
    """Returns (mean, worst) over output sentences. Range -1..1, higher = more consistent."""
    prem = split_sentences(premise_text)
    hyp = split_sentences(output_text)
    pairs = [(p, h) for h in hyp for p in prem]
    probs = []
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i + batch_size]
        enc = nli_tok([p for p, _ in batch], [h for _, h in batch], padding=True,
                      truncation="only_first", max_length=512, return_tensors="pt")
        probs.append(torch.softmax(nli_model(**enc).logits, dim=-1))
    probs = torch.cat(probs).view(len(hyp), len(prem), -1)
    ent = probs[..., ENT_IDX].max(dim=1).values      # best support per output sentence
    con = probs[..., CON_IDX].max(dim=1).values      # strongest contradiction per output sentence
    per_sent = ent - con
    return per_sent.mean().item(), per_sent.min().item()

def premise_src_def(s):
    """Original report + definitions of the gold CUIs: identical premise for every method."""
    defs = [doc_by_cui[c].page_content for c in s["cuis"] if c in doc_by_cui]
    return " ".join([s["jargon"]] + defs)

def score_summac(df, samples_by_id):
    out = df.copy()
    src_mean, src_worst, ref_mean = [], [], []
    for sid, ans in zip(df["sample_id"], df["answer"]):
        s = samples_by_id[sid]
        m, w = summac_zs(premise_src_def(s), ans)
        r, _ = summac_zs(s["lay"], ans)
        src_mean.append(m); src_worst.append(w); ref_mean.append(r)
    out["summac_src_def"] = src_mean
    out["summac_src_def_worst"] = src_worst
    out["summac_ref"] = ref_mean
    return out

c:\anaconda3\envs\kg-rag\lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


{0: 'SUPPORTS', 1: 'REFUTES', 2: 'NOT ENOUGH INFO'} -> entail 0 | contradict 1


In [14]:
# M1 -- SummaC for every answer. Crash-safe and resumable (see R0).
# A saved score is reused only if the answer text is identical (md5), so an un-cleaned answer is never reused.
# Full run: ~16,740 answers, roughly 8-12 h on CPU. Peak RAM: ~1 GB (NLI model)
import hashlib
assert "answer_raw" in scored.columns, "run P3 first"
md5 = lambda t: hashlib.md5(t.encode("utf-8")).hexdigest()

report_previous("summac")
saved = {(r["method"], r["sample_id"]): r for r in read_jsonl_safe(summac_file)}
todo = [(m, sid, a) for m, sid, a in zip(scored["method"], scored["sample_id"], scored["answer"])
        if saved.get((m, sid), {}).get("md5") != md5(a)]
total, n_have = len(scored), len(scored) - len(todo)
print(f"SummaC already saved: {n_have:,}/{total:,} | to score now: {len(todo):,}")

t_start, k, sid = time.time(), 0, None
out = SafeAppender(summac_file)
keep_awake(True)
write_status("summac", state="running", done=n_have, total=total, last_sample_id=None)
try:
    for k, (method, sid, ans) in enumerate(todo, start=1):
        s = samples_by_id[sid]
        m, w = summac_zs(premise_src_def(s), ans)
        r, _ = summac_zs(s["lay"], ans)
        rec = {"model": MODEL, "method": method, "sample_id": sid, "md5": md5(ans),
               "summac_src_def": m, "summac_src_def_worst": w, "summac_ref": r}
        out.write(rec)
        saved[(method, sid)] = rec
        if k % 10 == 0:
            write_status("summac", state="running", done=n_have + k, total=total, last_sample_id=sid)
        if k == 1 or k % 200 == 0 or k == len(todo):
            el = time.time() - t_start
            print(f"{time.strftime('%H:%M')} | saved {n_have + k:,}/{total:,} | {el/k:.1f} s/answer | "
                  f"~{(len(todo) - k) * el / k / 3600:.1f} h left")
    write_status("summac", state="finished", done=total, total=total, last_sample_id=sid)
except BaseException as e:
    why = "interrupted (Interrupt button)" if isinstance(e, KeyboardInterrupt) else f"error: {type(e).__name__}: {str(e)[:150]}"
    n_done = n_have + max(k - 1, 0)
    write_status("summac", state="stopped, " + why, done=n_done, total=total, last_sample_id=sid)
    print(f"\nSTOPPED: {why}\n{n_done:,}/{total:,} scores are safely saved. Restart & Run All continues from here.")
    raise
finally:
    out.close()
    keep_awake(False)

for col in ["summac_src_def", "summac_src_def_worst", "summac_ref"]:
    scored[col] = [saved[(m, sid)][col] for m, sid in zip(scored["method"], scored["sample_id"])]
display(scored.groupby("method")[["summac_src_def", "summac_src_def_worst", "summac_ref"]].mean().round(3))

[summac] first start
SummaC already saved: 60/60 | to score now: 0


,summac_src_def,summac_src_def_worst,summac_ref
method,,,
1hop,0.446,0.422,0.009
2hop,0.275,0.257,-0.035
hybrid,0.156,0.147,0.028
hybrid_anchored,0.553,0.536,0.006
no_rag,-0.031,-0.134,0.102
oracle,0.190,0.183,0.011


In [15]:
# E1 -- explanation metrics: readability + what happens to each gold_hard term. CPU, ~2 min. Peak RAM: ~50 MB
# term_kept / term_glossed / term_bare as in the pilot. The gloss rule was checked on 30 pilot answers
# (27/30 agree, 0 false positives), so term_glossed is a lower bound.
import re
import numpy as np
import textstat

def norm(text):
    """Words only: 'X-ray' -> ' x ray '. Used to FIND terms."""
    return " " + " ".join(re.findall(r"[a-z0-9]+", str(text).lower())) + " "

def norm_punct(text):
    """Words plus ( ) , - as tokens. Used only to SPOT an explanation right after a term."""
    return " " + " ".join(re.findall(r"[a-z0-9]+|[(),\-]", str(text).lower())) + " "

hard_spans = {}
with open(TEST_MENT, encoding="utf-8", newline="") as f:
    for r in csv.DictReader(f, delimiter="\t"):
        i = int(r["row"])
        if i in samples_by_id and r["chosen"] in samples_by_id[i]["gold_hard"]:
            hard_spans.setdefault(i, {}).setdefault(r["chosen"], set()).add(norm(r["span"]).strip())
ids = set(scored["sample_id"])
n_hard = sum(len(samples_by_id[i]["gold_hard"]) for i in ids)
n_span = sum(len(hard_spans.get(i, {})) for i in ids)
print(f"gold_hard CUIs in run: {n_hard} | with a caption span: {n_span}")
assert n_span == n_hard, "some gold_hard CUIs have no span; check the mentions file version"

GLOSS_AFTER = re.compile(r"^\s*(\(|,\s*(a|an|the|which|meaning|or|also called|called)\b|\s*-\s|\s+(is|are|means)\s+(a|an|the|when|where)\b)")

def glossed_at(a_punct, term):
    pattern = r"(?<= )" + r" (?:- )?".join(map(re.escape, term.split())) + r"(?= )"
    for m in re.finditer(pattern, a_punct):
        if GLOSS_AFTER.match(a_punct[m.end():]) or a_punct[:m.start()].rstrip().endswith("("):
            return True
    return False

assert glossed_at(norm_punct("a cloudy look (ground glass opacities)"), "ground glass opacities")
assert glossed_at(norm_punct("the x-ray (a picture of the chest)"), "x ray")
assert not glossed_at(norm_punct("there are ground glass opacities in the lung"), "ground glass opacities")

def term_scores(answer, sid):
    spans = hard_spans.get(sid)
    if not spans:
        return None, None, None
    a_words, a_punct = norm(answer), norm_punct(answer)
    kept = glossed = 0
    for variants in spans.values():
        found = [x for x in variants if " " + x + " " in a_words]
        if not found:
            continue
        kept += 1
        glossed += any(glossed_at(a_punct, x) for x in found)
    n = len(spans)
    return kept / n, glossed / n, (kept - glossed) / n

ts = [term_scores(a, i) for a, i in zip(scored["answer"], scored["sample_id"])]
scored["term_kept"], scored["term_glossed"], scored["term_bare"] = zip(*ts)
scored["fkgl"] = scored["answer"].map(lambda t: textstat.flesch_kincaid_grade(str(t)))
scored["fre"]  = scored["answer"].map(lambda t: textstat.flesch_reading_ease(str(t)))

src_fkgl = np.mean([textstat.flesch_kincaid_grade(samples_by_id[i]["jargon"]) for i in ids])
lay_fkgl = np.mean([textstat.flesch_kincaid_grade(samples_by_id[i]["lay"]) for i in ids])
print(f"FKGL source caption: {src_fkgl:.1f} | lay reference: {lay_fkgl:.1f}")
display(scored.groupby("method")[["fkgl", "fre", "term_kept", "term_glossed", "term_bare"]].mean().round(3))

gold_hard CUIs in run: 21 | with a caption span: 21
FKGL source caption: 11.6 | lay reference: 7.3


,fkgl,fre,term_kept,term_glossed,term_bare
method,,,,,
1hop,9.466,49.012,0.361,0.000,0.361
2hop,8.400,56.393,0.296,0.000,0.296
hybrid,8.892,55.358,0.472,0.000,0.472
hybrid_anchored,10.176,43.212,0.556,0.111,0.444
no_rag,5.535,76.885,0.333,0.000,0.333
oracle,8.463,54.781,0.574,0.111,0.463


## Summary

In [16]:
# S8 -- summary + files to send back. Peak RAM: ~0 MB
metric_cols = ["gold_cui_recall", "summac_src_def", "summac_src_def_worst", "summac_ref",
               "term_kept", "term_glossed", "term_bare", "fkgl", "fre",
               "n_docs", "context_words", "answer_words", "seconds"]
summary = scored.groupby(["model", "method"])[metric_cols].mean().round(3)
display(summary)

suffix = "_smoke" if TEST_SMOKE else ""
scored_csv  = OUT_DIR/f"test_kb_v2_{TAG}_scored{suffix}.csv"
summary_csv = OUT_DIR/f"test_kb_v2_{TAG}_summary{suffix}.csv"
scored.to_csv(scored_csv, index=False)
summary.to_csv(summary_csv)
print("send these files back:")
for p in [gen_file, summac_file, scored_csv, summary_csv, OUT_DIR/f"machine_{TAG}.json"]:
    print(f"  {p}  ({p.stat().st_size/1e6:.1f} MB)")

gold_cui_recall  summac_src_def  \
model        method                                             
qwen2.5:1.5b 1hop                       0.630           0.446   
             2hop                       0.685           0.275   
             hybrid                     0.194           0.156   
             hybrid_anchored            0.370           0.553   
             no_rag                       NaN          -0.031   
             oracle                     0.685           0.190   

                              summac_src_def_worst  summac_ref  term_kept  \
model        method                                                         
qwen2.5:1.5b 1hop                            0.422       0.009      0.361   
             2hop                            0.257      -0.035      0.296   
             hybrid                          0.147       0.028      0.472   
             hybrid_anchored                 0.536       0.006      0.556   
             no_rag                         -0.134       0.102      0.333   
             oracle                          0.183       0.011      0.574   

                              term_glossed  term_bare    fkgl     fre  n_docs  \
model        method                                                             
qwen2.5:1.5b 1hop                    0.000      0.361   9.466  49.012     4.1   
             2hop                    0.000      0.296   8.400  56.393     3.4   
             hybrid                  0.000      0.472   8.892  55.358     6.1   
             hybrid_anchored         0.111      0.444  10.176  43.212     5.8   
             no_rag                  0.000      0.333   5.535  76.885     0.0   
             oracle                  0.111      0.463   8.463  54.781     1.8   

                              context_words  answer_words  seconds  
model        method                                                 
qwen2.5:1.5b 1hop                      97.9          14.7    0.302  
             2hop                      98.3          14.7    0.279  
             hybrid                   103.5          14.3    0.556  
             hybrid_anchored          110.7          13.4    0.520  
             no_rag                     0.0          18.5   13.228  
             oracle                    56.0          14.3    0.274

send these files back:
  d:\Project\Radiology_RAG\results\test\test_kb_v2_qwen25_1.5b_generations.jsonl  (0.0 MB)
  d:\Project\Radiology_RAG\results\test\test_kb_v2_qwen25_1.5b_summac.jsonl  (0.0 MB)
  d:\Project\Radiology_RAG\results\test\test_kb_v2_qwen25_1.5b_scored_smoke.csv  (0.0 MB)
  d:\Project\Radiology_RAG\results\test\test_kb_v2_qwen25_1.5b_summary_smoke.csv  (0.0 MB)
  d:\Project\Radiology_RAG\results\test\machine_qwen25_1.5b.json  (0.0 MB)
